# Task Type Classifier v1 - TD-IDF + Logistic Regression
Predicts Task.type (academic, personal, health, social) from title + description text

In [22]:
# ==== Imports ====
import pandas as pd #loading/inspecting csv, od is universal alias
from sklearn.model_selection import train_test_split # stratified split
from sklearn.feature_extraction.text import TfidfVectorizer # tf-idf
from sklearn.linear_model import LogisticRegression # train model
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix # evaluate
import joblib

In [2]:
# ==== Data ====
# Reads csv into DataFrame
df = pd.read_csv("../data/tasks_labeled.csv")
# Shows first 4 rows
df.head()

,text,label
0,Finish calculus problem set,academic
1,Review Chapter 8 before tomorrow's quiz,academic
2,Submit chemistry lab report by Friday,academic
3,Email professor about project requirements,academic
4,Study for the Physics II midterm,academic


Split ratio + stratification

* 80/20 split
200 examples -> 160 train / 40 test

* Stratify = yes 
Forces the same class proportions in train and test
25/25/25/25%

In [3]:
# ==== Split ====
# Pull out single column as Series (1D array)
X = df["text"] 
y = df["label"]

# Splits features and labels the same, so they correspond after shuffling
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    stratify=y,     
    random_state=42  # "random" shuffle reproducible
)

print(f"Train: {len(X_train)}, Test: {len(X_test)}")

Train: 160, Test: 40


TD_IDF vectorizer settings

* ngram_range = (1, 1) single words (e.g. "lab", "report")
To not risk overfitting at this data size -> for v1

* No max_features -> use every word that appears
At current data size, total vocab won't exceed few hundred

* fit_transform on train only
Fit -> vectorizer learns the vocabulary, list of words and IDF weight based on training set
Transform -> converts test into numeric vector

* transform on test
Test set reuses vocab and IDF from train
No data leakage

* .shape -> output (# examples, # vocab words)

X will be a matrix feeding into softmax logistic regression. 
Row -> one task description
Column -> one word
Cell -> TF-IDF score

In [4]:
# ==== TF-IDF ====

vectorizer = TfidfVectorizer(ngram_range=(1, 1))

X_train_vec = vectorizer.fit_transform(X_train)
X_test_vec = vectorizer.transform(X_test)

print(f"Vocabulary size: {len(vectorizer.vocabulary_)}")
print(f"Train shape: {X_train_vec.shape}")

Vocabulary size: 408
Train shape: (160, 408)


Model Training
* scikit-learn -> LogisticRegression 
Softmax -> multiclass generalization of sigmoid
(aj) = (e^zj) / sum(e^zk)
where j is index of correct class

Loss = -log(aj)

Gradient descent for each parameter

* Checking convergence with max_iter=1000

* .fit(...) -> gradient descent runs here

In [5]:
model = LogisticRegression(max_iter=1000)
model.fit(X_train_vec, y_train)

,"max_iter max_iter: int, default=100Maximum number of iterations taken for the solvers to converge.",1000
,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add an L2 penalty term and it is the default choice;- `'l1'`: add an L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` and `C` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'`, `l1_ratio` set to any float between 0 and 1 for `penalty='elasticnet'`, and `C=np.inf` for `penalty=None`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0.0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation <regularized-logistic-loss>`) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*",None
,"random_state random_state: int, RandomState instance, default=NoneUsed when ``solver`` == 'sag', 'saga' or 'liblinear' to shuffle thedata. See :term:`Glossary <random_state>` for details.",None
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use in the optimization problem. Default is '

Predict and Evaluate

* .predict -> runs trained model on 40 tests

* classification_report
Precision: of everything the model classified, what was correct? Punish false positives
Recall: of everything that was X, what fraction did the model catch? Punish false negatives
f1-score: mean of precision and recall
Overall accuracy

* confusion_matrix
rows -> true labels
Columns -> Predicted labels

In [6]:
y_pred = model.predict(X_test_vec)

print(f"Accuracy: {accuracy_score(y_test, y_pred):.2%}")
print()
print(classification_report(y_test, y_pred))
print()
print(confusion_matrix(y_test, y_pred, labels=model.classes_))
print(model.classes_)

Accuracy: 65.00%

              precision    recall  f1-score   support

    academic       1.00      0.70      0.82        10
      health       0.46      0.60      0.52        10
    personal       0.83      0.50      0.62        10
      social       0.57      0.80      0.67        10

    accuracy                           0.65        40
   macro avg       0.72      0.65      0.66        40
weighted avg       0.72      0.65      0.66        40


[[7 1 0 2]
 [0 6 1 3]
 [0 4 5 1]
 [0 2 0 8]]
['academic' 'health' 'personal' 'social']


In [7]:
train_pred = model.predict(X_train_vec)
print(f"Train Accuracy: {accuracy_score(y_train, train_pred):.2%}")

Train Accuracy: 99.38%


In [8]:
results_df = pd.DataFrame({
    "text": X_test,
    "true_label": y_test,
    "predicted_label": y_pred
})

misclassified = results_df[results_df["true_label"] != results_df["predicted_label"]]
misclassified

,text,true_label,predicted_label
28,CS 350 milestone due next Monday,academic,health
90,Call mom back,personal,health
101,Go to the gym lol,health,social
97,Call my parents tonight,personal,health
84,Vacuum dorm,personal,social
20,Double-check my circuit analysis calculations,academic,social
113,mental health check-in,health,social
164,Text Alex about this weekend,social,health
22,Meet with project group after class,academic,social
106,Check in with my therapist,health,social


# Overfitting
(train accuracy 99.38%, test accuracy 65%)

* Cause: ratio between features and examples
    - tf-idf -> 408 vocab, model has the capacity to assign weight to words that only appear once or twice -> memorizing train set

    - misclassified test rows -> unique, rare words. Short vocab light phrases (no distinguishing words)

* Try Solution:
    - min_dif -> removes words that appear only once
    - Double regularization (C=0.5) -> penalized large weights (shrink values of parameters, but keeping them)

In [11]:
# New vectorizer obj for min_df
vectorizer_v2 = TfidfVectorizer(ngram_range=(1, 1), min_df=2)

X_train_vec_v2 = vectorizer_v2.fit_transform(X_train)
X_test_vec_v2 = vectorizer_v2.transform(X_test)

print(f"Vocabulary size (v2): {len(vectorizer_v2.vocabulary_)}")

# New model obj, each LogisticRegression instance holds one set of weights from one .fit()
model_v2 = LogisticRegression(max_iter=1000, C=0.5)
model_v2.fit(X_train_vec_v2, y_train)

train_pred_v2 = model_v2.predict(X_train_vec_v2)
test_pred_v2 = model_v2.predict(X_test_vec_v2)

print(f"Train Accuracy: {accuracy_score(y_train, train_pred_v2):.2%}")
print(f"Test Accuracy: {accuracy_score(y_test, test_pred_v2):.2%}")
print()
print(classification_report(y_test, test_pred_v2))

print(confusion_matrix(y_test, test_pred_v2, labels=model_v2.classes_))
print(model_v2.classes_)

Vocabulary size (v2): 138
Train Accuracy: 90.62%
Test Accuracy: 62.50%

              precision    recall  f1-score   support

    academic       1.00      0.70      0.82        10
      health       0.45      0.50      0.48        10
    personal       0.67      0.60      0.63        10
      social       0.54      0.70      0.61        10

    accuracy                           0.62        40
   macro avg       0.66      0.62      0.63        40
weighted avg       0.66      0.62      0.63        40

[[7 1 0 2]
 [0 5 2 3]
 [0 3 6 1]
 [0 2 1 7]]
['academic' 'health' 'personal' 'social']


The gap between train and test shrinked, so it's memorizing less. But test accuracy stayed almost flat. -> overfitting wasn't only problem.

* Cause: classes do overlap in vocab, so the remaining gap is a data problem.

* Try Solution: +15 new examples per class (keeping data balanced)

In [14]:
df_v3 = pd.read_csv("../data/tasks_labeled.csv")
print(f"Total examples: {len(df_v3)}")
print(df_v3["label"].value_counts())

X_v3 = df_v3["text"]
y_v3 = df_v3["label"]

X_train_v3, X_test_v3, y_train_v3, y_test_v3 = train_test_split(
    X_v3, y_v3,
    test_size=0.2,
    stratify=y_v3,
    random_state=42
)

vectorizer_v3 = TfidfVectorizer(ngram_range=(1, 1), min_df=2)
X_train_vec_v3 = vectorizer_v3.fit_transform(X_train_v3)
X_test_vec_v3 = vectorizer_v3.transform(X_test_v3)

print(f"Vocabulary size (v3): {len(vectorizer_v3.vocabulary_)}")

model_v3 = LogisticRegression(max_iter=1000, C=0.5)
model_v3.fit(X_train_vec_v3, y_train_v3)

train_pred_v3 = model_v3.predict(X_train_vec_v3)
test_pred_v3 = model_v3.predict(X_test_vec_v3)

print(f"Train Accuracy: {accuracy_score(y_train_v3, train_pred_v3):.2%}")
print(f"Test Accuracy: {accuracy_score(y_test_v3, test_pred_v3):.2%}")
print()
print(classification_report(y_test_v3, test_pred_v3))
print()
print(confusion_matrix(y_test_v3, test_pred_v3, labels=model_v3.classes_))
print(model_v3.classes_)

Total examples: 260
label
academic    65
personal    65
health      65
social      65
Name: count, dtype: int64
Vocabulary size (v3): 174
Train Accuracy: 91.35%
Test Accuracy: 51.92%

              precision    recall  f1-score   support

    academic       0.60      0.69      0.64        13
      health       0.43      0.23      0.30        13
    personal       0.40      0.46      0.43        13
      social       0.60      0.69      0.64        13

    accuracy                           0.52        52
   macro avg       0.51      0.52      0.50        52
weighted avg       0.51      0.52      0.50        52


[[9 3 0 1]
 [0 3 7 3]
 [4 1 6 2]
 [2 0 2 9]]
['academic' 'health' 'personal' 'social']


In [15]:
check_words = ["dermatology", "concussion", "dentist", "levothyroxine", "hearing"]
for w in check_words:
    print(w, "->", "IN vocabulary" if w in vectorizer_v3.vocabulary_ else "DROPPED")

dermatology -> DROPPED
concussion -> DROPPED
dentist -> DROPPED
levothyroxine -> DROPPED
hearing -> DROPPED


In [19]:
vectorizer_v3b = TfidfVectorizer(ngram_range=(1, 1), min_df=1)
X_train_vec_v3b = vectorizer_v3b.fit_transform(X_train_v3)
X_test_vec_v3b = vectorizer_v3b.transform(X_test_v3)

print(f"Vocabulary size (v3b, min_df=1): {len(vectorizer_v3b.vocabulary_)}")

model_v3b = LogisticRegression(max_iter=1000, C=0.5)
model_v3b.fit(X_train_vec_v3b, y_train_v3)

train_pred_v3b = model_v3b.predict(X_train_vec_v3b)
test_pred_v3b = model_v3b.predict(X_test_vec_v3b)

print(f"Train Accuracy: {accuracy_score(y_train_v3, train_pred_v3b):.2%}")
print(f"Test Accuracy: {accuracy_score(y_test_v3, test_pred_v3b):.2%}")
print()
print(confusion_matrix(y_test_v3, test_pred_v3b, labels=model_v3b.classes_))

Vocabulary size (v3b, min_df=1): 517
Train Accuracy: 98.08%
Test Accuracy: 65.38%

[[11  1  0  1]
 [ 0  5  5  3]
 [ 3  1  8  1]
 [ 1  1  1 10]]


## Summary

| Version | Data | min_df | C | Train Acc | Test Acc |
|   v1  | 200 rows | 1 | 1.0 | 99.38% | 65.00% |
|   v2  | 200 rows | 2 | 0.5 | 90.62% | 62.50% |
|   v3  | 260 rows | 2 | 0.5 | 91.35% | 51.92% |
| v3b (final) | 260 rows | 1 | 0.5 | 98.08% | 65.38% |

**Findings:**
- v1 shows clear overfitting (99.38% train vs 65.00% test): high variance, as expected with 408 TF-IDF features over only 160 training examples.
- v2's regularization (`min_df=2`, `C=0.5`): narrowed the train/test gap but *lowered* test accuracy, when test accuracy is what actually matters.
- v3 revealed an interaction: `min_df=2` combined with newly-added low-frequency-but-informative vocabulary (e.g. "dermatology," "concussion," "dentist") stripped the words meant to disambiguate overlapping classes, hurting performance.
-v3b reverting to `min_df=1` on the 260-row dataset: recovered performance, confirming the interaction and landing on the best test accuracy overall (65.38%).
- Remaining confusion is concentrated in personal/health/social, which share vocabulary (scheduling language, campus-activity language) by nature of the task, not a fixable modeling artifact — a reasonable ceiling for TF-IDF + Logistic Regression on ~260 examples.

In [20]:
# FINAL MODEL
vectorizer_final = TfidfVectorizer(ngram_range=(1, 1), min_df=1)
X_train_final = vectorizer_final.fit_transform(X_train_v3)
X_test_final = vectorizer_final.transform(X_test_v3)

model_final = LogisticRegression(max_iter=1000, C=0.5)
model_final.fit(X_train_final, y_train_v3)

print(f"Final Test Accuracy: {accuracy_score(y_test_v3, model_final.predict(X_test_final)):.2%}")

Final Test Accuracy: 65.38%


In [23]:
# Save artifacts
joblib.dump(model_final, "../models/task_classifier_v1_logreg.joblib")
joblib.dump(vectorizer_final, "../models/task_classifier_v1_vectorizer.joblib")

['../models/task_classifier_v1_vectorizer.joblib']

In [25]:
loaded_model = joblib.load("../models/task_classifier_v1_logreg.joblib")
loaded_vectorizer = joblib.load("../models/task_classifier_v1_vectorizer.joblib")

sample = ["Finish problem set 2 for calc1"]
sample_vec = loaded_vectorizer.transform(sample)
print(loaded_model.predict(sample_vec))
# predict_proba returns probability for all 4 classes
print(loaded_model.predict_proba(sample_vec))

['academic']
[[0.51700972 0.1923789  0.14561913 0.14499225]]
